# Bone fracture image classification (educational prototype)

**This is not a medical device. Do not use it for diagnosis, triage, treatment, or other clinical decisions.** No clinical validation, external validation, or performance claim is provided.

This notebook trains a small binary image-classification model from a separately obtained local dataset. It uses only the training split for optimization, the validation split for model selection, and evaluates the held-out test split once after training. Running it can overwrite the locally ignored model artifact described below.


## Data and reproducibility requirements

The dataset is never downloaded by this project. Set `BONE_FRACTURE_DATA` to a local directory containing exactly `train/`, `val/`, and `test/`; each split must contain `fractured/` and `not fractured/` with valid `.jpg`/`.jpeg` files directly inside. The loader rejects empty classes, malformed files, symlinks, unexpected entries, and byte-identical files shared across splits. The class-index mapping is fixed and stored in the model sidecar.

These checks **cannot establish patient/study-level split independence**, identify near-duplicates, verify labels, establish provenance/licensing/de-identification, or prove external validity. Verify these matters independently before any research use. Use only data you are authorized to process, and never commit images, patient information, model artifacts, or notebook outputs.


In [ ]:
from pathlib import Path
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from bone_fracture.artifacts import MODEL_FILENAME, write_model_manifest
from bone_fracture.data import CLASS_NAMES, load_datasets

SEED = 42
IMAGE_SIZE = (100, 100)
BATCH_SIZE = 32
EPOCHS = 20
DATA_ROOT = Path(os.environ.get("BONE_FRACTURE_DATA", "Dataset")).expanduser()
ARTIFACT_DIR = Path(os.environ.get("BONE_FRACTURE_ARTIFACT_DIR", "artifacts")).expanduser()

# Reproducible initialization and deterministic TensorFlow operations where supported.
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

datasets, split_counts = load_datasets(
    DATA_ROOT, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, seed=SEED
)
print("Validated image counts by split and class (aggregate counts only):")
for split_name, class_counts in split_counts.items():
    print(f"{split_name}: {class_counts}")
print("Class-to-index mapping:", {name: index for index, name in enumerate(CLASS_NAMES)})


In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3), name="xray_rgb"),
        tf.keras.layers.Rescaling(1.0 / 255, name="scale_pixels"),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", name="class_probabilities"),
    ],
    name="bone_fracture_cnn",
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)
training_process = model.fit(
    datasets["train"],
    validation_data=datasets["val"],
    epochs=EPOCHS,
    callbacks=[early_stopping],
)


In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / MODEL_FILENAME
pending_model_path = None
try:
    with tempfile.NamedTemporaryFile(
        prefix=".bone-fracture-pending-", suffix=".keras", dir=ARTIFACT_DIR, delete=False
    ) as temporary_model:
        pending_model_path = Path(temporary_model.name)
    model.save(pending_model_path)
    os.replace(pending_model_path, model_path)
finally:
    if pending_model_path is not None and pending_model_path.exists():
        pending_model_path.unlink()

best_epoch = int(np.argmin(training_process.history["val_loss"])) + 1
manifest_path = write_model_manifest(
    model_path,
    best_epoch=best_epoch,
    epochs_completed=len(training_process.history["val_loss"]),
    seed=SEED,
    image_size=IMAGE_SIZE,
)
print(f"Saved local model artifact: {model_path.name}")
print(f"Saved local schema/checksum sidecar: {manifest_path.name}")


## Final held-out test evaluation

The test split is used only in this final evaluation cell, after training and validation-based early stopping are complete. These aggregate values describe only this local run and are **not** clinical performance or external validation.


In [ ]:
test_metrics = model.evaluate(datasets["test"], return_dict=True, verbose=0)
print("Held-out test split aggregate metrics (research prototype only):")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(training_process.history["loss"], label="train")
axes[0].plot(training_process.history["val_loss"], label="validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(training_process.history["accuracy"], label="train")
axes[1].plot(training_process.history["val_accuracy"], label="validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend()
fig.suptitle("Training history only; not clinical validation")
fig.tight_layout()
